## MLP를 활용한 MNIST 분류 모델 구현 (PyTorch)

이번 실습에서는 PyTorch를 활용하여 MNIST 분류 모델을 구현하고 학습시켜 볼 것입니다.

아래 지시사항의 안내를 따라 코드를 완성하세요.

### 0. 라이브러리 불러오기

필요한 PyTorch 라이브러리를 불러옵니다.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

answer = {}

### 1. 데이터 탐색 및 전처리 (0점)

PyTorch의 `torchvision` 라이브러리를 사용하여 MNIST 데이터를 불러옵니다.

이때, 데이터를 Tensor 형태로 변환하는 전처리를 적용합니다.

In [5]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

# 1. 데이터 전처리 로직 정의
transform = ToTensor()

# Dataset을 불러오면서 전처리 로직을 적용합니다.
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

### 2. DataLoader 구성 (20점)

`None`을 아래 조건에 맞춰 DataLoader를 구성하는 코드로 대체해주세요.

먼저, 학습 데이터셋을 학습, 검증 데이터셋으로 분할합니다.
- torch.utils.data.random_split 함수를 사용하여 train_dataset을 학습 데이터와 검증 데이터로 분할합니다.
- 검증 데이터의 수는 10000개로 설정합니다.

그 다음, 각 데이터셋을 활용해서 학습, 검증, 테스트 DataLoader를 각각 구성합니다.
- 학습 데이터의 DataLoader는 train_dataset을 사용하고, batch_size는 32로 설정합니다. 학습 데이터는 `shuffle=True`로 섞어주는 것이 좋습니다.
- 검증 데이터의 DataLoader는 val_dataset을 사용하고, batch_size는 32로 설정합니다.
- 테스트 데이터의 DataLoader는 test_dataset을 사용하고, batch_size는 32로 설정합니다.
- 학습, 검증, 테스트 DataLoader를 각각 `train_loader`, `val_loader`, `test_loader` 변수에 저장합니다.

In [6]:
from torch.utils.data import DataLoader, random_split

val_size = 10000
train_size = len(train_dataset) - val_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# 2. DataLoader 구성
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
answer["Q1-1"] = len(train_data)
answer["Q1-2"] = len(val_data)
answer["Q1-3"] = len(test_dataset)

### 3-1. 모델 정의 (10점)

`None`을 다음 조건을 만족하는 MLP 모델을 정의하는 코드로 대체해주세요.
- 모델의 레이어는 총 3개입니다.
- 입력값의 차원은 28 * 28입니다.
- 은닉층의 차원은 순서대로 128, 64 입니다.
- 은닉층의 활성화 함수는 `ReLU`를 사용합니다.
- 출력값의 차원은 10입니다.
- 출력층에는 활성화 함수(`Softmax`)를 두지 않고 logits를 그대로 반환합니다. (손실 함수 `CrossEntropyLoss`가 내부적으로 softmax를 처리합니다.)

In [7]:
# 3-1. 모델 정의
# Note. Sequential 모델로 정의하지 마세요.
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x
model = MLP()

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
torch.save(model.state_dict(), 'mnist_model.pth')

### 3-2. 최적화 함수 구성 (10점)

아래의 조건을 만족하도록 최적화 함수를 구성해주세요.

- optimizer는 `Adam`를 사용합니다.
- loss는 `CrossEntropyLoss`를 사용합니다.
- 학습률은 0.001로 설정합니다.

In [8]:
# 3-2. 최적화 함수 구성
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
answer["Q3-2-A"] = str(type(criterion))
answer["Q3-2-B"] = optimizer.state_dict()

### 4. 모델 학습 (30점)
None을 지시사항의 조건을 만족하도록 모델을 학습시키는 코드로 대체해주세요.
- epochs는 10으로 설정합니다.
- 학습 데이터로 `train_loader`를 사용합니다.
- 검증 데이터로 `val_loader`를 사용합니다.

In [9]:
# 4-2. 모델 학습
epochs = 10
for epoch in range(epochs):
    model.train()
    # 학습 DataLoader에서 image와 labels를 불러와서 학습을 진행합니다.
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss = 0
    correct = 0
    # 검증 DataLoader에서 image와 labels를 불러와서 검증을 진행합니다.
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            val_loss += criterion(outputs, labels).item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
    # 검증 결과를 출력합니다.
    print(f"Epoch {epoch+1}, Validation Loss: {val_loss/len(val_loader):.4f}, Accuracy: {correct/len(val_loader.dataset):.4f}")
    
### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
answer["Q4"] = {'epochs': epochs}

Epoch 1, Validation Loss: 0.1648, Accuracy: 0.9509
Epoch 2, Validation Loss: 0.1147, Accuracy: 0.9649
Epoch 3, Validation Loss: 0.0923, Accuracy: 0.9725
Epoch 4, Validation Loss: 0.0836, Accuracy: 0.9728
Epoch 5, Validation Loss: 0.0886, Accuracy: 0.9748
Epoch 6, Validation Loss: 0.1019, Accuracy: 0.9714
Epoch 7, Validation Loss: 0.0810, Accuracy: 0.9764
Epoch 8, Validation Loss: 0.0864, Accuracy: 0.9779
Epoch 9, Validation Loss: 0.0884, Accuracy: 0.9786
Epoch 10, Validation Loss: 0.0977, Accuracy: 0.9767


### 5. 모델 평가 (30점)

테스트 데이터에 대해 모델을 평가하고, loss값을 `test_loss` 변수에, 테스트 정확도 값을 `test_acc` 변수에 저장해주세요. 두 값이 특정 기준을 만족하면 점수가 부여됩니다.

In [10]:
# 5. 모델 평가
model.eval()
test_loss = 0
correct = 0
# 테스트 DataLoader에서 image와 labels를 불러와서 테스트를 진행합니다.
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        test_loss += criterion(outputs, labels).item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
test_loss = test_loss / len(test_loader)
test_acc = correct / len(test_loader.dataset)

### 아래 코드는 채점용 코드입니다. 수정하면 안됩니다.
print(f"Test accuracy: {test_acc:.3f}")
answer["Q5-1"] = test_loss
answer["Q5-2"] = test_acc

Test accuracy: 0.978


### 제출

모든 문제를 해결하셨으면 아래 코드를 실행해서 결과를 저장한 후, 우측 상단의 '제출' 버튼을 눌러서 코드를 제출해주세요.

In [11]:
# Export data
import json
with open("submission.json", "w") as f:
    json.dump(answer, f)